In [ ]:
hugging_face_1bLLamaInstruct = "WRITE_YOUR_HF_TOKEN_HERE"

In [ ]:
from huggingface_hub import login

login(hugging_face_1bLLamaInstruct)

In [ ]:
# ============================================================================
# JAVA 21 SETUP - Add this BEFORE pip install pyserini
# ============================================================================

import os
import subprocess

print("Checking Java version...")

# Check current Java version
try:
    result = subprocess.run(['java', '-version'], capture_output=True, text=True)
    current_version = result.stderr
    print(f"Current Java:\n{current_version}")
except:
    print("Java not found")

# Install Java 21
print("\n📥 Installing Java 21...")
!sudo apt-get update -qq
!sudo apt-get install -y openjdk-21-jdk-headless -qq

# Set Java 21 as default
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]

# Verify installation
print("\n✓ Verifying Java 21 installation...")
!java -version

print("\n" + "="*70)
print("Java 21 installed! Now you can install pyserini.")
print("="*70)

In [ ]:
# ============================================================================
# INSTALL COMPATIBLE VERSIONS
# ============================================================================

print("Installing compatible package versions...")

# Install newer transformers that supports Qwen3
!pip install -q transformers>=4.46.0
!pip install -q pyserini==0.22.1

# Install other dependencies
!pip install -q ir_measures torch torchvision torchaudio sentence-transformers numpy

#install Faiss
!pip install faiss-cpu --no-cache

print("✓ All packages installed with compatible versions!")
print("\nVerifying installations...")

In [ ]:
import os
import torch
import numpy as np
import transformers
import pyserini
from collections import defaultdict
from pyserini.search.lucene import LuceneSearcher
from pyserini.index.lucene import IndexReader
import ir_measures
from ir_measures import MAP, nDCG, P
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from pyserini.encode import SpladeQueryEncoder
import warnings

# Suppress expected warnings
warnings.filterwarnings('ignore', message='Some weights of.*were not initialized')
warnings.filterwarnings('ignore', message='Some parameters are on the meta device')
warnings.filterwarnings('ignore', message='.*overflowing tokens are not returned.*')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================================
# GOOGLE DRIVE SETUP
# ============================================================================
from google.colab import drive, userdata
import os
import sys

# Check if running in Google Colab
try:
    IN_COLAB = True
    print("✓ Running in Google Colab")
except:
    IN_COLAB = False
    print("✓ Running locally")

# Mount Google Drive if in Colab
if IN_COLAB:
    print("\nMounting Google Drive...")
    drive.mount('/content/drive')
    print("✓ Google Drive mounted")

    # Set base directory
    BASE_DIR = '/content/drive/MyDrive/TEXT/FinalProject'

    if os.path.exists(BASE_DIR):
        print(f"✓ Found directory: {BASE_DIR}")
        os.chdir(BASE_DIR)
    else:
        print(f"⚠ Directory not found: {BASE_DIR}")
        search_paths = [
            '/content/drive/MyDrive/TEXT/FinalProject',
            '/content/drive/MyDrive/FinalProject',
            '/content/drive/MyDrive/TEST/FinalProject',
            '/content/drive/MyDrive',
        ]

        found = False
        for path in search_paths:
            query_file = os.path.join(path, 'queriesROBUST.txt')
            if os.path.exists(query_file):
                print(f"  ✓ Found files in: {path}")
                os.chdir(path)
                BASE_DIR = path
                found = True
                break
else:
    BASE_DIR = os.getcwd()
    print(f"Working directory: {BASE_DIR}")

# Verify files exist
print("\n📁 Checking for required files...")
print(f"Current directory: {os.getcwd()}")

if os.path.exists('queriesROBUST.txt'):
    with open('queriesROBUST.txt', 'r') as f:
        num_lines = len(f.readlines())
    print(f"  ✓ Found: queriesROBUST.txt ({num_lines} lines)")
else:
    print(f"  ✗ Missing: queriesROBUST.txt")

if os.path.exists('qrels_50_Queries'):
    with open('qrels_50_Queries', 'r') as f:
        num_lines = len(f.readlines())
    print(f"  ✓ Found: qrels_50_Queries ({num_lines} lines)")
else:
    print(f"  ✗ Missing: qrels_50_Queries")

print("\n" + "="*70)
if os.path.exists('queriesROBUST.txt') and os.path.exists('qrels_50_Queries'):
    print("✓ Setup complete! All required files found.")
else:
    print("⚠ WARNING: Missing required files.")
print("="*70)

# ROBUST04 RRF-Optimized Pipeline

## Key Improvements:
1. **Pure RRF Fusion** - Following Cormack 2009 paper recommendations
2. **Optimal k=60** - Standard parameter from paper
3. **Consistent rerank_depth=300** - For train/test consistency
4. **Improved RM3 parameters** - fb_docs=20, fb_terms=15, original_query_weight=0.6
5. **Parameter tuning experiment** - Find optimal k and rerank_depth

## Configuration

In [ ]:
# Model configuration
USE_MONOT5 = True

print(f"\n🎯 Ensemble Configuration (RRF-Optimized):")
print(f"  • MonoT5 Reranker: {USE_MONOT5}")
print(f"  • Fusion: Pure RRF (no manual weights)")
print(f"  • RRF k: 60 (Cormack 2009 recommendation)")
print(f"  • Rerank depth: 300 (consistent for train/test)")

## 2. Load Index and Configure Searchers

In [ ]:
print("Loading ROBUST04 index...")
index_reader = IndexReader.from_prebuilt_index('robust04')

# BM25 searcher
bm25_searcher = LuceneSearcher.from_prebuilt_index('robust04')
bm25_searcher.set_bm25(k1=0.9, b=0.4)

# RM3 searcher - OPTIMIZED PARAMETERS
rm3_searcher = LuceneSearcher.from_prebuilt_index('robust04')
rm3_searcher.set_bm25(k1=0.9, b=0.4)
rm3_searcher.set_rm3(fb_docs=20, fb_terms=15, original_query_weight=0.6)

searchers = [bm25_searcher, rm3_searcher]

print("✓ Configured BM25 + RM3")
print("  BM25: k1=0.9, b=0.4")
print("  RM3 (OPTIMIZED): fb_docs=20, fb_terms=15, original_query_weight=0.6")

## 3. Load Queries

In [ ]:
def load_queries(query_file):
    queries = {}
    with open(query_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(None, 1)
            if len(parts) == 2:
                queries[parts[0]] = parts[1]
    return queries

def load_qrels(qrel_file):
    qrels = defaultdict(dict)
    with open(qrel_file, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                qrels[parts[0]][parts[2]] = int(parts[3])
    return qrels

queries = load_queries('queriesROBUST.txt')
qrels = load_qrels('qrels_50_Queries')
train_qids = sorted(qrels.keys())
test_qids = sorted([q for q in queries if q not in qrels])

print(f"Loaded {len(queries)} queries ({len(train_qids)} train, {len(test_qids)} test)")

## 4. Helper Functions

In [ ]:
def reciprocal_rank_fusion(rank_lists, k=60):
    """
    Pure RRF implementation following Cormack et al. 2009.
    
    Formula: score(d) = Σ 1/(k + rank_i(d))
    
    Args:
        rank_lists: List of lists, where each inner list contains (docid, score) tuples
                   sorted by score desc (or just rank order).
        k: RRF constant (default=60 as per Cormack 2009)
    
    Returns:
        List of (docid, rrf_score) tuples sorted by RRF score descending
    """
    rrf_scores = defaultdict(float)

    for rank_list in rank_lists:
        for rank, (docid, _) in enumerate(rank_list):
            # Pure RRF: no weights, just 1/(k + rank)
            rrf_scores[docid] += 1.0 / (k + rank + 1)

    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

def classify_query(query_text):
    """Classify query by length"""
    wc = len(query_text.split())
    return 'short' if wc <= 3 else 'medium' if wc <= 6 else 'long'

## 5. Multi-Method Retrieval

In [ ]:
def multi_method_retrieval(query_text, k=1000):
    """
    Get results from BM25 and RM3, return both separately for RRF.
    """
    bm25_hits = searchers[0].search(query_text, k=k)
    rm3_hits = searchers[1].search(query_text, k=k)

    bm25_results = [(h.docid, h.score) for h in bm25_hits]
    rm3_results = [(h.docid, h.score) for h in rm3_hits]

    return bm25_results, rm3_results

## 6. Load Reranking Models

In [ ]:
# MonoT5 - Discriminative Reranker
monot5_model, monot5_tokenizer = None, None
if USE_MONOT5:
    print("Loading MonoT5 Reranker...")

    # Clear GPU memory
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    try:
        mn = "castorini/monot5-base-msmarco-10k"
        monot5_tokenizer = AutoTokenizer.from_pretrained(mn)
        monot5_model = AutoModelForSeq2SeqLM.from_pretrained(mn).to(device)
        monot5_model.eval()
        print(f"  ✓ MonoT5 loaded successfully")
    except Exception as e:
        print(f"  MonoT5 load failed: {e}")
        USE_MONOT5 = False

## 7. MonoT5 Reranker

In [ ]:
def rerank_monot5(query, doc_ids, batch_size=8):
    """MonoT5 reranking (pointwise scoring)

    Returns dict of {doc_id: relevance_probability}
    Higher score = more relevant
    """
    if not USE_MONOT5 or monot5_model is None:
        return None

    doc_texts, valid_ids = [], []
    for did in doc_ids:
        try:
            doc = index_reader.doc(did)
            if doc:
                raw = doc.raw()
                if raw:
                    # Truncate document text
                    doc_texts.append(raw[:2000])
                    valid_ids.append(did)
        except Exception as e:
            pass

    if not doc_texts:
        return None

    # MonoT5 template
    prompts = [f"Query: {query} Document: {d} Relevant:" for d in doc_texts]

    # Token IDs
    true_token_id = monot5_tokenizer.encode("true")[0]
    false_token_id = monot5_tokenizer.encode("false")[0]

    all_scores = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        try:
            inputs = monot5_tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512
            ).to(device)

            with torch.no_grad():
                outputs = monot5_model.generate(
                    **inputs,
                    max_new_tokens=1,
                    return_dict_in_generate=True,
                    output_scores=True
                )
                logits = outputs.scores[0]
                true_logits = logits[:, true_token_id]
                false_logits = logits[:, false_token_id]
                probs = torch.softmax(torch.stack([false_logits, true_logits], dim=1), dim=1)
                batch_scores = probs[:, 1].cpu().tolist()  # Probability of "true"
                all_scores.extend(batch_scores)
        except Exception as e:
            print(f"  Batch error: {e}")
            all_scores.extend([0.5] * len(batch))

    return dict(zip(valid_ids, all_scores))

## 8. Pure RRF Pipeline (OPTIMIZED)

In [ ]:
def pure_rrf_pipeline(query, rerank_depth=300, k_rrf=60):
    """
    Pure RRF fusion: BM25 + RM3 + MonoT5 (NO manual weights!)
    
    Following Cormack et al. 2009:
    - k=60 (standard)
    - Equal treatment of all systems
    - Simple and effective
    
    Args:
        query: Query text
        rerank_depth: Number of docs to rerank with MonoT5
        k_rrf: RRF constant (default=60)
    
    Returns:
        List of (docid, rrf_score, rank) tuples
    """
    # Stage 1: Get BM25 and RM3 rankings
    bm25_results, rm3_results = multi_method_retrieval(query)
    
    # Build rank lists for RRF
    rank_lists = []
    
    # Add BM25 rankings (limit to rerank_depth for fair comparison)
    rank_lists.append(bm25_results[:rerank_depth])
    
    # Add RM3 rankings
    rank_lists.append(rm3_results[:rerank_depth])
    
    # Stage 2: Get MonoT5 ranking for top docs
    if USE_MONOT5 and monot5_model is not None:
        top_k_docids = [docid for docid, _ in bm25_results[:rerank_depth]]
        monot5_scores = rerank_monot5(query, top_k_docids)
        
        if monot5_scores and len(monot5_scores) > 0:
            # Sort by MonoT5 score and add to rank lists
            sorted_by_monot5 = sorted(monot5_scores.items(), key=lambda x: x[1], reverse=True)
            rank_lists.append(sorted_by_monot5)
    
    # Stage 3: Apply pure RRF fusion (k=60, no weights)
    fused_results = reciprocal_rank_fusion(rank_lists, k=k_rrf)
    
    # Build final results
    results = []
    for rank, (docid, score) in enumerate(fused_results[:1000], 1):
        results.append((str(docid), float(score), int(rank)))
    
    return results

## 9. Parameter Tuning Experiment

This cell runs experiments to find the optimal k value and rerank_depth.

We'll test:
- k values: [30, 40, 50, 60, 70, 80]
- rerank_depth values: [100, 200, 300, 400, 500]

**Note:** This will take time! Run this once to find optimal parameters.

In [ ]:
import time
import pandas as pd

# PARAMETER TUNING EXPERIMENT
RUN_PARAMETER_TUNING = False  # Set to True to run experiments

if RUN_PARAMETER_TUNING:
    print("="*70)
    print("🔬 PARAMETER TUNING EXPERIMENT")
    print("="*70)
    print("Testing combinations of k and rerank_depth on training set...")
    print()
    
    # Parameter ranges to test
    k_values = [30, 40, 50, 60, 70, 80]
    rerank_depths = [100, 200, 300, 400, 500]
    
    results_table = []
    
    for k_val in k_values:
        for depth in rerank_depths:
            print(f"\n{'='*70}")
            print(f"Testing k={k_val}, rerank_depth={depth}")
            print(f"{'='*70}")
            
            train_results = []
            start_time = time.time()
            
            for i, qid in enumerate(train_qids, 1):
                query_text = queries[qid]
                
                if i % 10 == 0:
                    print(f"  [{i}/{len(train_qids)}] Processing query {qid}...")
                
                try:
                    results = pure_rrf_pipeline(query_text, rerank_depth=depth, k_rrf=k_val)
                    
                    for docid, score, rank in results:
                        train_results.append(ir_measures.ScoredDoc(
                            query_id=str(qid),
                            doc_id=str(docid),
                            score=float(score)
                        ))
                except Exception as e:
                    print(f"  Error on query {qid}: {str(e)[:50]}")
                    continue
            
            # Convert qrels
            qrels_list = []
            for qid, doc_rels in qrels.items():
                for docid, rel in doc_rels.items():
                    qrels_list.append(ir_measures.Qrel(
                        query_id=str(qid),
                        doc_id=str(docid),
                        relevance=int(rel)
                    ))
            
            # Calculate metrics
            metrics = [MAP, nDCG@10, nDCG@20, P@10, P@20]
            results_metrics = ir_measures.calc_aggregate(metrics, qrels_list, train_results)
            
            elapsed = time.time() - start_time
            
            # Store results
            result_row = {
                'k': k_val,
                'rerank_depth': depth,
                'MAP': results_metrics[MAP],
                'nDCG@10': results_metrics[nDCG@10],
                'nDCG@20': results_metrics[nDCG@20],
                'P@10': results_metrics[P@10],
                'P@20': results_metrics[P@20],
                'time_s': elapsed
            }
            results_table.append(result_row)
            
            print(f"\n  Results: MAP={results_metrics[MAP]:.4f}, nDCG@10={results_metrics[nDCG@10]:.4f}")
            print(f"  Time: {elapsed:.1f}s")
    
    # Display results table
    df_results = pd.DataFrame(results_table)
    df_results = df_results.sort_values('MAP', ascending=False)
    
    print("\n" + "="*70)
    print("📊 PARAMETER TUNING RESULTS (sorted by MAP)")
    print("="*70)
    print(df_results.to_string(index=False))
    
    # Best configuration
    best_row = df_results.iloc[0]
    print("\n" + "="*70)
    print("🏆 BEST CONFIGURATION")
    print("="*70)
    print(f"  k = {int(best_row['k'])}")
    print(f"  rerank_depth = {int(best_row['rerank_depth'])}")
    print(f"  MAP = {best_row['MAP']:.4f}")
    print(f"  nDCG@10 = {best_row['nDCG@10']:.4f}")
    print(f"  Time = {best_row['time_s']:.1f}s")
    print("="*70)
    
    # Save results to CSV
    df_results.to_csv('parameter_tuning_results.csv', index=False)
    print("\n✓ Results saved to parameter_tuning_results.csv")
else:
    print("⚠️  Parameter tuning skipped (RUN_PARAMETER_TUNING=False)")
    print("   Set RUN_PARAMETER_TUNING=True to run experiments")
    print("   Using default: k=60, rerank_depth=300")

## 10. Evaluate on Training Set

In [ ]:
print("="*70)
print("📊 EVALUATING ON TRAINING SET (50 QUERIES)")
print("="*70)
print("Using OPTIMIZED parameters:")
print("  • Pure RRF (no manual weights)")
print("  • k = 60")
print("  • rerank_depth = 300")
print("  • RM3: fb_docs=20, fb_terms=15, original_query_weight=0.6")
print("="*70)
print()

import time

# Generate results for training queries
train_results = []
start_time = time.time()

for i, qid in enumerate(train_qids, 1):
    query_text = queries[qid]
    query_type = classify_query(query_text)

    print(f"[{i}/{len(train_qids)}] Query {qid} ({query_type}): {query_text[:50]}...")

    try:
        # Use pure RRF pipeline with k=60, rerank_depth=300
        results = pure_rrf_pipeline(query_text, rerank_depth=300, k_rrf=60)

        for docid, score, rank in results:
            train_results.append(ir_measures.ScoredDoc(
                query_id=str(qid),
                doc_id=str(docid),
                score=float(score)
            ))

        print(f"  ✓ Retrieved {len(results)} docs")
    except Exception as e:
        print(f"  ✗ Error: {str(e)[:100]}")
        continue

total_time = time.time() - start_time

# Convert qrels to ir_measures format
qrels_list = []
for qid, doc_rels in qrels.items():
    for docid, rel in doc_rels.items():
        qrels_list.append(ir_measures.Qrel(
            query_id=str(qid),
            doc_id=str(docid),
            relevance=int(rel)
        ))

# Calculate metrics
print("\n" + "="*70)
print("📈 COMPUTING METRICS...")
print("="*70)

metrics = [MAP, nDCG@10, nDCG@20, P@10, P@20]
results_metrics = ir_measures.calc_aggregate(metrics, qrels_list, train_results)

print("\n" + "="*70)
print("🏆 TRAINING SET PERFORMANCE (50 queries)")
print("="*70)
print("\n📊 Your Scores:")
print(f"  MAP:      {results_metrics[MAP]:.4f}  ← Main metric for competition")
print(f"  nDCG@10:  {results_metrics[nDCG@10]:.4f}")
print(f"  nDCG@20:  {results_metrics[nDCG@20]:.4f}")
print(f"  P@10:     {results_metrics[P@10]:.4f}")
print(f"  P@20:     {results_metrics[P@20]:.4f}")

print(f"\n🔧 Configuration:")
print(f"  Method: Pure RRF (BM25 + RM3 + MonoT5)")
print(f"  k: 60")
print(f"  rerank_depth: 300")

print(f"\n🎯 Expected Performance Ranges:")
print(f"  Pure RRF (MonoT5 + RM3): MAP 0.28-0.35")

print(f"\n📈 Your Performance Assessment:")
if results_metrics[MAP] >= 0.32:
    print(f"  🌟 EXCELLENT! Very strong performance!")
    print(f"     You're in the competitive range (0.32+).")
elif results_metrics[MAP] >= 0.28:
    print(f"  ✓ GOOD! Solid performance!")
    print(f"     Consider: Add more retrieval signals (SPLADE, dense retrieval)")
elif results_metrics[MAP] >= 0.24:
    print(f"  ⚠️  Below target range")
    print(f"     Run parameter tuning experiment to find better k/depth")
else:
    print(f"  ⚠️  Significantly below expected")
    print(f"     Check: Are all models loaded correctly?")

print(f"\n⏱️  Evaluation time: {total_time:.1f}s ({total_time/len(train_qids):.1f}s per query)")

print("\n" + "="*70)
print("📝 IMPROVEMENTS FROM ORIGINAL:")
print("="*70)
print("• Pure RRF (no manual weights) - more robust")
print("• k=60 (Cormack 2009 recommendation)")
print("• Consistent rerank_depth=300 (train & test)")
print("• Improved RM3 parameters (fb_docs=20, fb_terms=15)")
print("="*70)

## 11. Generate Test Set Results

In [ ]:
import time

print("="*70)
print("🚀 GENERATING RUN 3 - PURE RRF OPTIMIZED")
print("="*70)
print(f"Configuration:")
print(f"  • Method: Pure RRF (BM25 + RM3 + MonoT5)")
print(f"  • k = 60 (Cormack 2009)")
print(f"  • rerank_depth = 300")
print(f"  • RM3: fb_docs=20, fb_terms=15, original_query_weight=0.6")
print(f"  • Total test queries: {len(test_qids)}")
print(f"\n🎯 Expected Performance:")
print(f"  Pure RRF Ensemble: MAP 0.28-0.35")
print("="*70)
print()

run3_results = []
start_time = time.time()

for i, qid in enumerate(test_qids, 1):
    query_text = queries[qid]
    query_type = classify_query(query_text)

    print(f"[{i}/{len(test_qids)}] Query {qid} ({query_type}): {query_text[:70]}...")

    query_start = time.time()

    # Run pure RRF pipeline
    results = pure_rrf_pipeline(query_text, rerank_depth=300, k_rrf=60)

    query_time = time.time() - query_start

    # Add results
    for docid, score, rank in results:
        run3_results.append({
            'qid': str(qid),
            'docid': str(docid),
            'rank': int(rank),
            'score': float(score)
        })

    print(f"  ✓ Retrieved {len(results)} docs in {query_time:.1f}s")

    # Progress report every 10 queries
    if i % 10 == 0:
        elapsed = time.time() - start_time
        avg_time = elapsed / i
        remaining = (len(test_qids) - i) * avg_time

        print()
        print("─"*70)
        print(f"📊 PROGRESS REPORT: {i}/{len(test_qids)} queries ({i/len(test_qids)*100:.1f}%)")
        print("─"*70)
        print(f"  Time:")
        print(f"    • Elapsed: {elapsed/60:.1f} min")
        print(f"    • Remaining: {remaining/60:.1f} min")
        print(f"    • Avg per query: {avg_time:.1f}s")
        print(f"  Results so far:")
        print(f"    • Total rankings: {len(run3_results):,}")
        print(f"    • Avg docs/query: {len(run3_results)/i:.1f}")
        if torch.cuda.is_available():
            print(f"  GPU:")
            print(f"    • Current: {torch.cuda.memory_allocated()/1e9:.2f} GB")
            print(f"    • Peak: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
        print("─"*70)
        print()

total_time = time.time() - start_time

print()
print("="*70)
print("✓ GENERATION COMPLETE!")
print("="*70)
print(f"Summary:")
print(f"  • Queries processed: {len(test_qids)}")
print(f"  • Total rankings: {len(run3_results):,}")
print(f"  • Avg docs/query: {len(run3_results)/len(test_qids):.1f}")
print(f"  • Total time: {total_time/60:.1f} minutes")
print(f"  • Avg time/query: {total_time/len(test_qids):.1f}s")
if torch.cuda.is_available():
    print(f"  • Peak GPU memory: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
print("="*70)
print()

## 12. Save Results

In [ ]:
with open('run_3_rrf_optimized.res', 'w') as f:
    for r in run3_results:
        score = r['score']
        if isinstance(score, (list, np.ndarray)):
            score = float(score[0]) if len(score) > 0 else 0.0
        else:
            score = float(score)
        f.write(f"{r['qid']} Q0 {r['docid']} {r['rank']} {score:.6f} run3_rrf_optimized\n")

print("✓ Saved to run_3_rrf_optimized.res")
print("\nFirst 5 lines:")
with open('run_3_rrf_optimized.res', 'r') as f:
    for i, line in enumerate(f):
        if i < 5:
            print(line.strip())
        else:
            break

print("\n" + "="*70)
print("📝 SUBMISSION READY!")
print("="*70)
print("File: run_3_rrf_optimized.res")
print("\nKey improvements over original:")
print("  1. Pure RRF fusion (no manual weights)")
print("  2. Optimal k=60 (Cormack 2009 paper)")
print("  3. Consistent rerank_depth=300")
print("  4. Better RM3 parameters")
print("  5. Parameter tuning framework included")
print("\nExpected improvement: +0.03 to +0.08 MAP")
print("(From 0.2738 → 0.30-0.35 range)")
print("="*70)